# 🟠 Project 09: Apache Kafka Fundamentals & Interactive Stream Processing

> **Data Engineering Diaries** | by Sathiya Murthy  
> Production-grade Apache Kafka patterns for real-time event streaming pipelines.

---

## 📋 Overview
This interactive notebook guides you through the fundamental building blocks of Apache Kafka in Data Engineering:
1. **Event Generation**: Realistic e-commerce, clickstream, and IoT data generator.
2. **Producer Patterns**: Batching, LZ4 compression, acks=all, and delivery callbacks.
3. **Consumer Groups**: Dynamic partition consumption, manual offset commits, and rebalance listeners.
4. **Stream Processing**: Stateful tumbling window aggregations and alert triggers.
5. **Dead Letter Queue (DLQ)**: Poison message handling and error isolation.


In [ ]:
import sys
import os
import json
from pprint import pprint

# Ensure project root is in sys.path
sys.path.append(os.path.abspath(".."))

from src.producer import EventGenerator, KafkaEventProducer, CompressionType
from src.consumer import KafkaEventConsumer, StreamProcessor
from src.admin import TopicManager, LocalSchemaRegistry, SALES_EVENT_AVRO_SCHEMA
from src.patterns import DeadLetterQueueHandler, NonRetryableException, TransactionalStreamProcessor

print("✅ Kafka Engineering Modules successfully loaded!")


## 1. Synthetic Event Data Generation

In production stream engineering, testing pipelines requires synthetic data that accurately models real-world cardinality, skew, and edge cases.
Here we initialize `EventGenerator` to generate sales, clickstream, and IoT readings.


In [ ]:
gen = EventGenerator(seed=42)

sample_sale = gen.generate_sale()
sample_click = gen.generate_click()
sample_iot = gen.generate_iot_reading()

print("--- Sample Sale Event Envelope ---")
pprint(sample_sale)

print("\n--- Sample IoT Reading ---")
pprint(sample_iot)


## 2. Producer Batching, Compression, & Metrics

We initialize `KafkaEventProducer` with `CompressionType.LZ4` and `use_mock=True` to simulate high-throughput producing.


In [ ]:
producer = KafkaEventProducer(
    compression=CompressionType.LZ4,
    delivery_guarantee="at_least_once",
    use_mock=True
)

sales_batch = [gen.generate_sale() for _ in range(100)]
msg_ids = producer.produce_batch(
    topic="sales-events",
    events=sales_batch,
    key_extractor=lambda e: e["user_id"]
)

print(f"Produced {len(msg_ids)} events into topic 'sales-events'")
pprint(producer.get_metrics())
producer.close()


## 3. Real-Time Stream Processor & Tumbling Window Aggregation

Here we test stateful stream processing:
1. Filtering cancelled order events.
2. Categorizing sales amounts (`Small`, `Medium`, `Large`, `Enterprise`).
3. Aggregating 1-minute tumbling window sums per region.


In [ ]:
from datetime import datetime, timezone
from src.consumer.kafka_consumer import ProcessedMessage

processor = StreamProcessor(window_size_sec=60)

# Feed sample events into stream processor
for i, event in enumerate(sales_batch[:10]):
    msg = ProcessedMessage(
        topic="sales-events",
        partition=0,
        offset=i,
        key=event["user_id"],
        value={"payload": event},
        headers={},
        timestamp=datetime.now(timezone.utc),
        raw_value_bytes=b""
    )
    enriched = processor.process_sales_event(msg)
    if enriched:
        print(f"Processed Order {enriched['order_id']} -> Category: {enriched['order_category']} Region: {enriched['region']}")

# Emit tumbling window results
results = processor.emit_aggregations()
print(f"\nEmitted {len(results)} windowed aggregation records:")
for r in results:
    pprint(r.to_dict())
